In [7]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = "llama3.2:3b",
    temperature=0.1
)
response = llm.invoke("Hola, funcionas en local?")
print(response.content)

¡Hola! Me alegra que hayas intentado contactarme. Como soy un modelo de lenguaje basado en inteligencia artificial, no tengo una ubicación física específica, por lo que puedo funcionar desde cualquier lugar del mundo.

Sin embargo, si necesitas ayuda con algo relacionado a tu localidad o región, puedo tratar de proporcionarte información y recursos relevantes. ¿En qué puedo ayudarte hoy?


In [12]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import JSONLoader

# cargamos la knowledge
folder_loader = DirectoryLoader(
    "../knowledge",
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader
)
docs_md = folder_loader.load()

# cargamos los json
def json_metadata(record: dict, metadata: dict):
    metadata["traceId"] = record.get("traceId", "NONE")
    metadata["service"] = record.get("service", "NONE")
    metadata["level"] = record.get("level", "NONE")

    return metadata

json_arguments = {
    "jq_schema": '.[] | select(has("error_message"))',
    "content_key": "error_message",
    "metadata_func": json_metadata
}

json_loader = DirectoryLoader(
    "../datasets",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs=json_arguments
)
docs_json = json_loader.load()

print(f"Archivos .md cargados: {len(docs_md)}")
print(f"Archivos JSON cargados: {len(docs_json)}")

Archivos .md cargados: 2
Archivos JSON cargados: 2279


In [23]:
docs = docs_md + docs_json

# ahora con los datos completos los pasamos por el textsplitter y preparamos el vectorstore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

embedding = OllamaEmbeddings(
    model = "nomic-embed-text"
)

chunks = text_splitter.split_documents(docs)

vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate